In [3]:
#Вариант 5
import pandas as pd
import numpy as np

df_events_records = pd.read_json(
    "events_records.json",
    orient="columns"
).T

df_events_records.columns = [
    "event_id", "user_id", "event_time", "event_type", "page"
]

df_events_records["event_id"] = df_events_records["event_id"].astype(int)
df_events_records["user_id"] = df_events_records["user_id"].astype(int)
df_events_records["event_time"] = pd.to_datetime(
    df_events_records["event_time"]
)


df_events_lines = pd.read_json("events_lines.json", lines=True)

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

df_events["event_time"] = pd.to_datetime(df_events["event_time"])


df_events = df_events.sort_values(
    by=["user_id", "event_time"]
).reset_index(drop=True)

time_diff = df_events.groupby("user_id")["event_time"].diff()

df_events["new_session"] = (
    time_diff.isna() | (time_diff > pd.Timedelta(minutes=30))
)

df_events["session_id"] = (
    df_events.groupby("user_id")["new_session"].cumsum()
)

print("Подготовленные данные о событиях:")
print(df_events.head())

print("\nРазмер таблицы событий:")
print(df_events.shape)

Подготовленные данные о событиях:
   event_id  user_id          event_time event_type         page  new_session  \
0      1407     1001 2024-01-01 06:25:31   purchase  /product/30         True   
1       819     1001 2024-01-05 07:12:56  page_view   /product/7         True   
2      1930     1001 2024-01-08 15:40:51      click            /         True   
3      1498     1001 2024-01-09 06:39:26     scroll  /product/12         True   
4       404     1001 2024-01-15 00:00:38  page_view  /product/13         True   

   session_id  
0           1  
1           2  
2           3  
3           4  
4           5  

Размер таблицы событий:
(4000, 7)


In [4]:
session_stats = (
    df_events.groupby(["user_id", "session_id"])
    .agg(events_in_session=("event_id", "count"))
    .reset_index()
)

user_avg_events = (
    session_stats.groupby("user_id")["events_in_session"]
    .mean()
)

print("\nЗадание 1. Среднее количество событий в сессии для каждого пользователя:")
print(user_avg_events)


Задание 1. Среднее количество событий в сессии для каждого пользователя:
user_id
1001    1.080000
1002    1.025000
1003    1.000000
1004    1.057143
1005    1.000000
          ...   
1096    1.022222
1097    1.000000
1098    1.000000
1099    1.000000
1100    1.000000
Name: events_in_session, Length: 100, dtype: float64


In [5]:
threshold = user_avg_events.quantile(0.95)

anomalous_users = user_avg_events[user_avg_events > threshold]

print("\nЗадание 2. Пользователи с аномально большим средним числом событий:")
print(anomalous_users)

print("\nПороговое значение (95-й перцентиль):")
print(threshold)


Задание 2. Пользователи с аномально большим средним числом событий:
user_id
1001    1.080000
1004    1.057143
1010    1.060606
1040    1.062500
1061    1.102564
Name: events_in_session, dtype: float64

Пороговое значение (95-й перцентиль):
1.0542084942084942


In [6]:
anomalous_users.to_csv("anomalous_users.csv", header=["Average_Events_Per_Session"])

print("\nЗадание 3. Список пользователей сохранен в anomalous_users.csv")


Задание 3. Список пользователей сохранен в anomalous_users.csv
